# Retail Stock Insight — EDA and Feature Engineering

This notebook walks through the synthetic retail stock dataset, cleaning, analysis, and feature engineering steps.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

project_root = Path('..') / 'Retail_Stock_Insight_Project'
raw_dir = project_root / 'data' / 'raw'
cleaned_dir = project_root / 'data' / 'cleaned'
fig_dir = project_root / 'outputs' / 'figures'

transactions = pd.read_csv(raw_dir / 'transactions.csv')
products = pd.read_csv(raw_dir / 'products.csv')
stores = pd.read_csv(raw_dir / 'stores.csv')
inventory = pd.read_csv(raw_dir / 'inventory.csv')
external = pd.read_csv(raw_dir / 'external_factors.csv')
transactions.head()

## Data overview
The dataset includes transactions, products, stores, inventory, and external demand factors.

In [ ]:
for label, df in {'transactions': transactions, 'products': products, 'stores': stores, 'inventory': inventory, 'external_factors': external}.items():
    print(label, df.shape)
    print(df.head().to_string(index=False))
    print()

## Data quality review
The notebook validates missing values, duplicates, invalid prices, and inventory mismatches.

In [ ]:
quality = pd.read_csv(cleaned_dir / 'data_quality_report.csv')
quality

## Join and feature engineering
Transactions are joined to product, store, and external factor tables to create a business-ready analytical dataset.

In [ ]:
trans = pd.read_csv(cleaned_dir / 'cleaned_transactions.csv')
trans['date'] = pd.to_datetime(trans['date'])
trans['net_revenue'] = trans['quantity'] * trans['selling_price']
trans = trans.merge(pd.read_csv(cleaned_dir / 'cleaned_products.csv'), on='product', how='left')
trans = trans.merge(pd.read_csv(cleaned_dir / 'cleaned_stores.csv'), on='store', how='left')
trans = trans.merge(pd.read_csv(cleaned_dir / 'cleaned_external_factors.csv'), left_on=['date','city'], right_on=['date','city'], how='left')
trans.head()

## Key business calculations
The chart and summary below show category revenue, store performance, and promotion effect.

In [ ]:
category_summary = trans.groupby('category')['net_revenue'].sum().sort_values(ascending=False)
print(category_summary.head())
print('Best category:', category_summary.idxmax())
print('Total revenue:', trans['net_revenue'].sum())

In [ ]:
plt.figure(figsize=(10,5))
category_summary.head(10).plot(kind='bar')
plt.title('Revenue by category')
plt.ylabel('Net revenue')
plt.tight_layout(); plt.savefig(fig_dir / 'revenue_by_category_demo.png', dpi=200); plt.close()
print('Saved demo figure')